# 4 · Ejercicio 3: listas de cinco formas (35 min)

El Ejercicio 3 casi siempre pide **la misma función** implementada de varias formas. Aquí verás las cinco con **un solo ejemplo**, para que se vea que son traducciones unas de otras.

**Ejemplo:** `evens(l)` devuelve los números pares de `l`, en el mismo orden.

### Preámbulo
Ejecuta esta celda antes que las demás: son las mismas definiciones que trae el examen, comentadas.

In [1]:
// ─────────────────────────────────────────────────────────────────────────────
// PREÁMBULO — las mismas definiciones que te dan en el examen.
// Ejecuta esta celda antes que ninguna otra del notebook.
// ─────────────────────────────────────────────────────────────────────────────

// Librería de tests. `run(TestX(f))` ejecuta los tests de un apartado.
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

// "no P" se representa como una función que, si le das una P, produce Nothing
// (un tipo SIN valores = una contradicción). Si existe, es que P es imposible.
type Not[P] = P => Nothing
// "P si y solo si Q" = un par de funciones: de P a Q y de Q a P.
type <=>[P, Q] = (P => Q, Q => P)

// Tercio excluso: para cualquier P, o tengo una P o tengo una prueba de que no hay P.
// Se usa escribiendo  lem[X]  y haciendo match sobre el Either que devuelve.
trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

// Doble negación: si "no es verdad que no P", entonces P.
// Se usa escribiendo  dn[X](nx => ...)  cuando la meta es X.
trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

// Plantilla de los isomorfismos: tú implementas from y to; las leyes las usan los tests.
trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)   // ir y volver deja igual una A
    def law2(b: B): Boolean = equalB(from(to(b)), b)   // ir y volver deja igual una B

// Árbol binario: o vacío, o un nodo con hijo izquierdo, raíz e hijo derecho.
enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

// Atajos para escribir árboles en los tests. El orden de los argumentos imita el dibujo:
// left(hijo, raiz) pone el hijo a la izquierda; right(raiz, hijo) a la derecha.
object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)

import Tree._   // para escribir leaf(1) en vez de Tree.leaf(1)

import $ivy.$                                

import org.scalatest.{Filter => _, _}, flatspec._, matchers._

// "no P" se representa como una función que, si le das una P, produce Nothing
// (un tipo SIN valores = una contradicción). Si existe, es que P es imposible.

defined type Not
defined type <=>
defined trait ExcludedMiddle
defined trait DoubleNegation
defined trait Isomorphic
defined class Tree
defined object Tree
import Tree._   // para escribir leaf(1) en vez de Tree.leaf(1)


In [2]:
// Así son los tests del examen: una clase que RECIBE tu función como parámetro.
// El tipo del parámetro (List[Int] => List[Int]) te dice la signatura que debes implementar.
// Cada  x shouldBe y  es un ejemplo de lo que debe devolver: léelos antes de programar.
class TestEvens(evens: List[Int] => List[Int]) extends AnyFlatSpec with should.Matchers:
    "evens" should "work" in:
        evens(List()) shouldBe List()
        evens(List(1, 2, 3, 4, 6)) shouldBe List(2, 4, 6)
        evens(List(1, 3)) shouldBe List()

defined class TestEvens

## Forma 1 · Recursión (divide y vencerás)

1. ¿Qué devuelvo para `Nil`?
2. Para `head :: tail`: **supón que ya tienes la solución del resto** (`tailSol`). ¿Cómo la combino con `head`?

In [3]:
def evens(l: List[Int]): List[Int] =
    l match
        // CASO BASE. Pregunta: ¿cuáles son los pares de una lista vacía? Ninguno.
        case Nil => Nil

        // CASO RECURSIVO. La lista tiene un primer elemento (head) y un resto (tail).
        case head :: tail =>
            // Supongo que la función YA funciona para el resto (que es una lista más corta).
            // Así que tailSol = los pares de tail, en orden. No hace falta pensar cómo.
            val tailSol = evens(tail)
            // Solo me queda decidir qué hago con head:
            //   si es par, va delante de los pares del resto (delante, porque va antes en la lista)
            //   si no, la respuesta es directamente la del resto
            if head % 2 == 0 then head :: tailSol else tailSol

run(TestEvens(evens))

cell2$Helper$TestEvens:
evens
- should work


defined function evens

## Forma 2 · `foldRight` (traducción directa de la forma 1)

`l.foldRight(LO_DE_NIL)((head, tailSol) => LO_DEL_CASO_RECURSIVO)`

| Recursión | `foldRight` |
|---|---|
| `case Nil => X` | primer argumento: `X` |
| cuerpo de `case head :: tail` | la función `(head, tailSol) => ...` |
| `evens(tail)` | `tailSol` |

In [4]:
// foldRight hace por nosotros el "l match / case Nil / case head :: tail / llamada recursiva".
// Solo hay que darle las dos piezas que cambian de un problema a otro:
def evens(l: List[Int]): List[Int] =
    l.foldRight(
        // 1ª pieza: la respuesta para la lista vacía (lo que iba en  case Nil => ...).
        // Le pongo el tipo (: List[Int]) porque un Nil "a secas" Scala lo entiende como
        // una lista sin tipo de elemento, y después no dejaría hacer head :: tailSol.
        Nil: List[Int]
    )(
        // 2ª pieza: cómo combinar la cabeza con la solución del resto.
        // Es literalmente la última línea de la forma 1. foldRight nos pasa tailSol ya
        // calculado, así que NO hay que llamar a evens(tail).
        (head, tailSol) => if head % 2 == 0 then head :: tailSol else tailSol
    )

run(TestEvens(evens))

cell2$Helper$TestEvens:
evens
- should work


defined function evens

## Forma 3 · Recursión final (patrón iterativo)

Es un bucle: un acumulador `out` que empieza con un valor y se actualiza con cada elemento. La llamada recursiva es **lo último** que se hace.

⚠️ Al añadir con `::` recorriendo de izquierda a derecha, la lista queda **al revés** → `.reverse` al final.

In [5]:
def evens(l: List[Int]): List[Int] =

    // Función auxiliar con un ACUMULADOR (out) donde voy guardando la solución parcial,
    // y aux = lo que me queda por recorrer. Es como un bucle while:
    //   out = valor inicial;  mientras queden elementos: actualizar out;  devolver out
    def step(out: List[Int], aux: List[Int]): List[Int] =
        aux match
            // No queda nada por recorrer: la solución parcial ya es la final.
            case Nil => out
            case head :: tail =>
                // Actualizo out con head y sigo con el resto. La llamada a step es lo
                // ÚLTIMO que se hace (no queda nada pendiente después): eso es "recursión
                // final", y evita desbordar la memoria con listas muy largas.
                // Añado con :: (delante) porque es instantáneo; añadir al final es lento.
                step(if head % 2 == 0 then head :: out else out, tail)

    // Arranco con el acumulador vacío y la lista entera.
    // Como cada elemento se puso DELANTE del anterior, la lista sale al revés:
    // para List(2, 4, 6) out acaba siendo List(6, 4, 2). Por eso el .reverse final.
    step(Nil, l).reverse

run(TestEvens(evens))

cell2$Helper$TestEvens:
evens
- should work


defined function evens

## Forma 4 · `foldLeft` (traducción directa de la forma 3)

`l.foldLeft(VALOR_INICIAL)((out, head) => NUEVO_OUT)` y luego el retoque final.

⚠️ El orden es `(acumulado, elemento)`, **al revés** que en `foldRight`.

In [ ]:
// foldLeft es el bucle de la forma 3 ya escrito. Le doy:
//   1) el valor inicial del acumulador (lo que pasaba en step(Nil, l))
//   2) cómo actualizar el acumulador con cada elemento (lo que pasaba en step(..., tail))
def evens(l: List[Int]): List[Int] =
    l.foldLeft(Nil: List[Int])(
        // ¡Atención al orden! Primero el acumulado (out), luego el elemento (head).
        // En foldRight era al revés: (head, tailSol).
        (out, head) => if head % 2 == 0 then head :: out else out
    ).reverse       // el mismo retoque final que en la forma 3, por la misma razón

run(TestEvens(evens))

## Forma 5 · Sin recursión: funciones de orden superior

| Función | Hace |
|---|---|
| `map(f)` | transforma cada elemento |
| `filter(p)` | se queda con los que cumplen `p` |
| `flatMap(f)` | cada elemento da una lista, y se juntan |
| `exists(p)` / `forall(p)` | ¿alguno / todos cumplen? |
| `zipWithIndex`, `zip`, `sum`, `reverse`… | ver preámbulo del examen |

In [ ]:
// "Quedarse con los que cumplen una condición" es exactamente lo que hace filter.
// _ % 2 == 0 es la condición: una función Int => Boolean (equivale a x => x % 2 == 0).
def evens(l: List[Int]): List[Int] =
    l.filter(_ % 2 == 0)

// Con flatMap: cada elemento produce UNA LISTA, y flatMap junta todas.
// Si x es par produce List(x) (se queda); si no, Nil (desaparece).
// Es útil cuando te obligan a usar flatMap, o cuando hay que filtrar y transformar a la vez.
def evens2(l: List[Int]): List[Int] =
    l.flatMap(x => if x % 2 == 0 then List(x) else Nil)

run(TestEvens(evens))
run(TestEvens(evens2))

## Dos trucos que salen mucho

**1. Usar `case` dentro de los folds** cuando hay que mirar la forma de los datos. Cada caso recibe el par `(head, tailSol)` (o `(out, head)` en `foldLeft`):

In [ ]:
// split: separa los Left y los Right de una lista en dos listas.
def split[A, B](l: List[Either[A, B]]): (List[A], List[B]) =
    // La solución es un PAR de listas, así que el valor inicial es un par de listas vacías
    // (con su tipo dicho, por la misma razón que antes).
    l.foldRight((List[A](), List[B]())):
        // En vez de (head, tailSol) => ..., escribo case para poder mirar DENTRO de head
        // (¿es Left o Right?) y de tailSol (desempaquetarlo en sus dos listas as y bs).
        // Cada case recibe el par (head, tailSol).
        case (Left(a), (as, bs))  => (a :: as, bs)   // un Left va a la primera lista
        case (Right(b), (as, bs)) => (as, b :: bs)   // un Right va a la segunda

split(List(Left(1), Right('a'), Left(2)))

**2. Acumulador con información extra (tupla).** Si para decidir necesitas la posición, el elemento anterior, un contador… guárdalo junto a la solución y al final quédate con lo que importa (`._1`).

In [ ]:
// Elementos en posiciones pares (0, 2, 4...). Para decidir si me quedo con un elemento
// necesito saber su posición, pero foldLeft solo me da (acumulado, elemento).
// Solución: el acumulado es un PAR (resultado parcial, posición actual).
def evenPositions[A](l: List[A]): List[A] =
    l.foldLeft((List[A](), 0)):            // empiezo: resultado vacío, posición 0
        case ((out, i), e) =>              // desempaqueto el acumulado y miro el elemento e
            // nuevo resultado: añado e solo si su posición es par
            // nueva posición: la siguiente (i + 1), pase lo que pase
            (if i % 2 == 0 then e :: out else out, i + 1)
    ._1.reverse     // ._1 = me quedo solo con el resultado (la posición ya no me sirve)
                    // .reverse = porque, como siempre con foldLeft y ::, salió al revés

evenPositions(List('a', 'b', 'c', 'd', 'e'))     // List(a, c, e)

## Mini-ejercicio (10 min): `countIf`

`countIf(p, l)` cuenta cuántos elementos de `l` cumplen `p`. Hazlo de las cinco formas (la quinta con `filter` y `length`).

In [ ]:
class TestCount(countIf: (Int => Boolean, List[Int]) => Int) extends AnyFlatSpec with should.Matchers:
    "countIf" should "work" in:
        countIf(_ > 0, List()) shouldBe 0
        countIf(_ > 0, List(1, -2, 3)) shouldBe 2
        countIf(_ % 2 == 0, List(2, 4, 6)) shouldBe 3

In [ ]:
def countIf[A](p: A => Boolean, l: List[A]): Int =
    ???

run(TestCount(countIf))

<details>
<summary><b>Ver solución</b></summary>

```scala
// 1. Recursión: base = 0 (en una lista vacía no hay nada que contar).
//    Recursivo: lo que cuente el resto, +1 si head cumple p.
def countIf[A](p: A => Boolean, l: List[A]): Int =
    l match
        case Nil => 0
        case head :: tail => (if p(head) then 1 else 0) + countIf(p, tail)

// 2. foldRight: las mismas dos piezas (0 y la combinación), con tailSol en lugar de la llamada.
def countIf[A](p: A => Boolean, l: List[A]): Int =
    l.foldRight(0)((head, tailSol) => if p(head) then tailSol + 1 else tailSol)

// 3. Recursión final: el acumulador out es "cuántos llevo contados".
//    Aquí no hace falta reverse: el resultado es un número, no una lista.
def countIf[A](p: A => Boolean, l: List[A]): Int =
    def step(out: Int, aux: List[A]): Int =
        aux match
            case Nil => out
            case head :: tail => step(if p(head) then out + 1 else out, tail)
    step(0, l)

// 4. foldLeft: mismo valor inicial y misma actualización. Orden (out, head).
def countIf[A](p: A => Boolean, l: List[A]): Int =
    l.foldLeft(0)((out, head) => if p(head) then out + 1 else out)

// 5. HOFs: "contar los que cumplen" = quedarse con los que cumplen y medir la lista.
def countIf[A](p: A => Boolean, l: List[A]): Int =
    l.filter(p).length
```

</details>

**Siguiente:** `05-ejercicio4-patrones.ipynb`